# Homework 10 - Mini Project

目标：完成一个小分类项目。不是追求高精度，而是能解释：数据 -> score -> loss -> grad -> update -> boundary。

In [ ]:
from pathlib import Path
import sys, math, random


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True


def check_or_todo(condition, message):
    if not condition:
        print(message)
        return False
    return True

## 完整例子 - 先手写一个 score

先不用 MLP。假设边界是 `score = x1 - x2`，score 大于 0 就判为红点。

In [ ]:
def simple_score(point):
    x1, x2 = point
    return x1 - x2

for point in [(-2, -1), (2, 1)]:
    s = simple_score(point)
    print(point, 'score=', s, 'label=', 1 if s > 0 else -1)

## Modify - 改模型结构后算参数数量

`MLP(2,[4,1])` 有 17 个参数。如果隐藏层改成 3 个神经元，就少几个？

In [ ]:
student_param_count_mlp_2_3_1 = None


def qa_check_10_modify():
    if not todo_guard([student_param_count_mlp_2_3_1]): return
    assert student_param_count_mlp_2_3_1 == 13
    print('OK: Modify 通过。')

qa_check_10_modify()

## 完整数据和画图工具

In [ ]:
from micrograd.engine import Value
from micrograd.nn import MLP

xs = [(-2,-1), (-1,-2), (-2,1), (-1,1), (1,2), (2,1), (1,-1), (2,-2), (0.5,1.5), (-1.5,-0.5)]
ys = [-1, -1, -1, -1, 1, 1, 1, 1, 1, -1]


def write_points_svg(points, labels, path='mini_project_points.svg'):
    width = height = 360
    def sx(x): return 40 + (x + 3) / 6 * 280
    def sy(y): return 320 - (y + 3) / 6 * 280
    items = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">', '<rect width="100%" height="100%" fill="white"/>']
    for (x,y), label in zip(points, labels):
        color = '#e74c3c' if label == 1 else '#3498db'
        items.append(f'<circle cx="{sx(x):.1f}" cy="{sy(y):.1f}" r="6" fill="{color}"/>')
    items.append('</svg>')
    Path(path).write_text('\n'.join(items), encoding='utf-8')
    return path

print('data svg:', write_points_svg(xs, ys))

## 作业 1-4 - 项目函数骨架

In [ ]:
random.seed(7)
model = MLP(2, [4, 1])


def predict_score(xrow):
    # TODO 1: 把 xrow 里的数字变成 Value，再喂给 model
    pass


def predict_label(xrow):
    # TODO 2: score > 0 返回 1，否则返回 -1
    pass


def hinge_loss(score, y):
    # TODO 3: relu(1 - y*score)
    pass


def total_loss():
    # TODO 4: 对所有样本求平均 hinge loss
    pass


def accuracy():
    # TODO 5: 返回预测正确的比例
    pass


def qa_check_10_project_functions():
    try:
        s = predict_score(xs[0])
        label = predict_label(xs[0])
        L = total_loss()
        acc = accuracy()
    except Exception as exc:
        print('请继续实现项目函数。当前:', type(exc).__name__, exc)
        return
    if any(v is None for v in [s, label, L, acc]):
        print('请继续实现 TODO：有函数返回 None。')
        return
    assert hasattr(s, 'data'), 'predict_score 应该返回 Value。'
    assert label in [-1, 1]
    assert hasattr(L, 'data'), 'total_loss 应该返回 Value。'
    assert 0 <= acc <= 1
    print('OK: 项目函数骨架通过。loss=', round(L.data, 4), 'acc=', round(acc, 3))

qa_check_10_project_functions()

## 作业 5 - 训练循环

In [ ]:
def train(steps=30, learning_rate=0.05):
    history = []
    for step in range(steps):
        # TODO 6: zero_grad
        # TODO 7: forward/loss
        # TODO 8: backward
        # TODO 9: update
        pass
    return history


def qa_check_10_train():
    try:
        history = train(steps=20, learning_rate=0.05)
    except Exception as exc:
        print('请继续实现 train。当前:', type(exc).__name__, exc)
        return
    if not history:
        print('train 应该返回 loss history。')
        return
    assert history[-1] <= history[0], '训练后 loss 应该不升高。'
    print('OK: 训练循环通过。first/last=', round(history[0], 4), round(history[-1], 4))

qa_check_10_train()

<details><summary>Show / Hide 提示</summary>

训练循环顺序：清零参数 grad -> 算 loss -> `loss.backward()` -> 对每个参数执行 `p.data -= learning_rate * p.grad`。

</details>

## Debug Lab - 项目里最常见的假通过

In [ ]:
# Debug 1：loss 下降但 accuracy 没变，可能正常吗？填 True/False
student_loss_down_acc_same_possible = None

# Debug 2：如果参数 data 完全没变，最可能漏了哪一步？填 'update'
student_missing_step_when_params_same = None

# Debug 3：如果 grad 一轮比一轮大但代码没报错，优先检查什么？填 'zero_grad'
student_first_check_when_grad_accumulates = None


def qa_check_10_debug():
    values = [student_loss_down_acc_same_possible, student_missing_step_when_params_same, student_first_check_when_grad_accumulates]
    if not todo_guard(values): return
    assert student_loss_down_acc_same_possible is True
    assert student_missing_step_when_params_same == 'update'
    assert student_first_check_when_grad_accumulates == 'zero_grad'
    print('OK: Mini Project Debug Lab 通过。')

qa_check_10_debug()

## 提交清单

- [ ] `qa_check_10_project_functions` 通过
- [ ] `qa_check_10_train` 通过
- [ ] Debug Lab 通过
- [ ] 我能解释一个样本如何从输入变成 loss，再变成参数更新

<details><summary>Show / Hide 答案</summary>

答案不要一开始看。正确答案应该能由完整例子和 Modify 题一步推出；如果你需要直接看答案，说明前一个台阶还没踩稳。

</details>